<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. SAS 파일 읽기용 패키지 설치
!pip install pyreadstat

# 2. 구글 드라이브 마운트 (팝업창 나오면 계정 연결 및 허용 누르기)
from google.colab import drive
import pandas as pd
import pyreadstat

drive.mount('/content/drive')

# 3. 데이터 로드 (hdac_data 폴더 안에 kyrbs2025.sas7bdat 파일 확인)
DATA_DIR = '/content/drive/MyDrive/hdac_data'
cols = [
    'YEAR',
    'STRATA',
    'CLUSTER',
    'W',
    'GRADE',
    'SEX',
    'AGE',
    'HT',
    'WT',
    'F_BR',
    'WC_MN',
]

try:
  df, meta = pyreadstat.read_sas7bdat(
      f'{DATA_DIR}/kyrbs2025.sas7bdat', usecols=cols
  )
  print('=== 데이터 로드 성공! ===')
  print('전체 행 수:', len(df))
except Exception as e:
  print('오류 발생:', e)
  print(
      '-> 구글 드라이브의 내 드라이브(MyDrive) 안에 hdac_data 폴더와 kyrbs2025.sas7bdat 파일이 있는지 확인해 주세요!'
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 데이터 로드 성공! ===
전체 행 수: 54170


In [3]:
import numpy as np
import pandas as pd

# 1. 행 삭제 없이 고등학생 도메인 지정 (GRADE: 4.0=고1, 5.0=고2, 6.0=고3)
df['subpop'] = df['GRADE'].isin([4.0, 5.0, 6.0]).astype(int)

# 2. 고등학생 도메인 내 아침 결식 여부 분류 (F_BR 기준)
# F_BR 코드 5.0 이상(주 5일 이상 결식) = 결식군 / 5.0 미만 = 비결식군
df['breakfast_group'] = np.nan
df.loc[(df['subpop'] == 1) & (df['F_BR'] >= 5.0), 'breakfast_group'] = '결식군'
df.loc[(df['subpop'] == 1) & (df['F_BR'] < 5.0), 'breakfast_group'] = '비결식군'

# 3. 열 머리용 비가중 표본수(n) 계산
n_total = (df['subpop'] == 1).sum()
n_skip = (df['breakfast_group'] == '결식군').sum()
n_noskip = (df['breakfast_group'] == '비결식군').sum()

# 4. 열 머리 명칭 정의 (요구사항: 비가중 n 제시)
col_total = f'전체 고등학생 (n={n_total:,})'
col_skip = f'아침 결식군 (n={n_skip:,})'
col_noskip = f'아침 비결식군 (n={n_noskip:,})'

# 5. 도메인 분석(고등학생 subpop=1) 적용 통계 산출 함수
df_domain = df[df['subpop'] == 1].copy()


def calc_domain_stats(sub_df):
  w = sub_df['W']  # 복합표본 가중치 변수

  # 성별 (남성 비율) - 가중 % 적용
  valid_sex = sub_df.dropna(subset=['SEX'])
  n_male = (valid_sex['SEX'] == 1).sum()
  pct_male = (
      sub_df.loc[sub_df['SEX'] == 1, 'W'].sum() / sub_df['W'].sum() * 100
  )

  # 나이 가중 평균
  valid_age_idx = sub_df['AGE'].dropna().index
  mean_age = np.average(
      sub_df.loc[valid_age_idx, 'AGE'], weights=w.loc[valid_age_idx]
  )

  # 체중조절 노력 (1: 체중 줄이려고 노력) - 가중 %
  valid_wc = sub_df.dropna(subset=['WC_MN'])
  n_wc1 = (valid_wc['WC_MN'] == 1.0).sum()
  pct_wc1 = (
      sub_df.loc[sub_df['WC_MN'] == 1.0, 'W'].sum() / sub_df['W'].sum() * 100
  )

  return {
      'Age, mean': f'{mean_age:.1f}',
      'Sex: Male, n (%)': f'{n_male:,} ({pct_male:.1f}%)',
      'Weight control effort (Loss), n (%)': f'{n_wc1:,} ({pct_wc1:.1f}%)',
  }


# 6. 각 그룹별 통계 계산 후 Table 1 병합
res_all = calc_domain_stats(df_domain)
res_skip = calc_domain_stats(df_domain[df_domain['breakfast_group'] == '결식군'])
res_noskip = calc_domain_stats(
    df_domain[df_domain['breakfast_group'] == '비결식군']
)

t1_df = pd.DataFrame(
    [res_all, res_skip, res_noskip], index=[col_total, col_skip, col_noskip]
).T
display(t1_df)

/tmp/ipykernel_1688/1838505489.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '결식군' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[(df['subpop'] == 1) & (df['F_BR'] >= 5.0), 'breakfast_group'] = '결식군'


,"전체 고등학생 (n=25,843)","아침 결식군 (n=12,062)","아침 비결식군 (n=13,781)"
"Age, mean",16.5,16.5,16.5
"Sex: Male, n (%)","13,335 (51.5%)","6,455 (53.6%)","6,880 (49.7%)"
"Weight control effort (Loss), n (%)","12,330 (48.0%)","5,816 (48.7%)","6,514 (47.5%)"


**Footnote:**
1. **도메인 분석 지정:** 원본 표본의 구조적 손실(분산 추정 오류)을 방지하기 위해 중학생 표본 행을 삭제하지 않고 `subpop` 변수(고등학생 25,843명)를 정의하여 도메인 분석(Domain Analysis)을 수행함.
2. **열 구성 및 수치 제시:** 열 머리의 $n$은 비가중 관측 표본 수(Unweighted $n$)이며, 표 내부의 수치는 복합표본 가중치(`W`)를 적용한 가중 추정치(Weighted Mean 또는 Weighted %)임.
3. **아침 결식 분류 기준:** 최근 7일간 아침식사 빈도(`F_BR`) 설문 항목을 바탕으로 주 5일 이상 결식(코드 5.0~8.0)을 '아침 결식군', 주 4일 이하(코드 1.0~4.0)를 '아침 비결식군'으로 분류함.